# 🧠 Transformer Components — Combined Notebook

All 14 transformer-component solutions from `torch/transformer/` in one notebook.

**Sections (in order):**

1. 01 · Scaled Dot-Product Attention
2. 02 · Single-Head Attention
3. 03 · Self-Attention
4. 04 · Multi-Head Attention (MHA)
5. 05 · Multi-Query Attention (MQA)
6. 06 · Grouped-Query Attention (GQA)
7. 07 · LayerNorm
8. 08 · RMSNorm
9. 09 · Residual Connection
10. 10 · Feed-Forward Network
11. 11 · Sinusoidal Positional Embedding
12. 12 · Rotary Positional Embedding (RoPE)
13. 13 · KV Cache
14. 14 · Transformer Decoder Block

> 💡 Each section is self-contained. Run the cells top-to-bottom within a section. Re-running across sections will redefine classes (that's fine).



---

# 01 · Scaled Dot-Product Attention

*Source: `torch/transformer/01-Scaled-Dot-Product-Attention/scaled-dot-product-attention.ipynb`*

---


# Implement Attention from Scratch
### Problem Statement
Implement a **Scaled Dot-Product Attention** mechanism from scratch using PyTorch. Your mission (should you choose to accept it) is to replicate what PyTorch's built-in `scaled_dot_product_attention` does — manually. This core component is essential in Transformer architectures and helps models focus on relevant parts of a sequence. You'll test your implementation against PyTorch's native one to ensure you nailed it.


### Requirements
1. **Define the Function**:
   - Create a function `scaled_dot_product_attention(q, k, v, mask=None)` that:
     - Computes attention scores via the dot product of query and key vectors.
     - Scales the scores using the square root of the key dimension.
     - Applies an optional mask to the scores.
     - Applies softmax to convert scores into attention weights.
     - Uses these weights to compute a weighted sum of values (V).

2. **Test Your Work**:
   - Use sample tensors for query (Q), key (K), and value (V).
   - Compare the result of your custom implementation with PyTorch's `F.scaled_dot_product_attention` using an `assert` to check numerical accuracy.


### Constraints
- ❌ Do NOT use `F.scaled_dot_product_attention` inside your custom function — that defeats the whole point.
- ✅ Your implementation must handle **batch dimensions** correctly.
- ✅ Support optional **masking** for future tokens or padding.
- ✅ Use only PyTorch ops — no cheating with external attention libs.



<details>
  <summary>💡 Hint</summary>
  Use `torch.matmul()` to compute dot products and `F.softmax()` for the final attention weights. The mask (if used) should be applied **before** the softmax using `masked_fill`.
</details>


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
import torch

torch.manual_seed(42)

batch_size = 1
seq_len = 3
dim = 3

q = torch.randn(batch_size, seq_len, dim)
k = torch.randn(batch_size, seq_len, dim)
v = torch.randn(batch_size, seq_len, dim)

In [ ]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Compute the scaled dot-product attention.
    
    Args:
        q: Query tensor of shape (..., seq_len_q, d_k)
        k: Key tensor of shape (..., seq_len_k, d_k)
        v: Value tensor of shape (..., seq_len_k, d_v)
        mask: Optional mask tensor of shape (..., seq_len_q, seq_len_k)
    
    Returns:
        output: Attention output tensor of shape (..., seq_len_q, d_v)
        attention_weights: Attention weights tensor of shape (..., seq_len_q, seq_len_k)
    """
    d_k = q.shape[-1]  # Get the last dimension size (key dimension)
    
    # Compute the dot product of Q and K^T
    scores = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))
    
    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Apply softmax to get attention weights along the last dimension
    attention_weights = F.softmax(scores, dim=-1)  # dim=-1 ensures softmax is applied across the last axis
    
    # Compute output by weighting V with the attention weights
    output = torch.matmul(attention_weights, v)
    
    return output, attention_weights

In [ ]:
# Testing on data & compare
output_custom, _ = scaled_dot_product_attention(q, k, v)
print(output_custom)
output = F.scaled_dot_product_attention(q, k, v)
print(output)

assert torch.allclose(output_custom, output, atol=1e-08, rtol=1e-05) # Check if they are close enough.



---

# 02 · Single-Head Attention

*Source: `torch/transformer/02-Single-Head-Attention/single-head-attention.ipynb`*

---


# Implement Single-Head Attention from Scratch
### Problem Statement
Single-Head Attention is the foundational building block of Multi-Head Attention. Instead of splitting the input into multiple heads, a single attention head computes attention over the **full** `d_model` dimension.

Your goal is to implement Single-Head Attention from scratch using PyTorch — projecting Q, K, V through linear layers, computing scaled dot-product attention, and producing the output.

---

### Requirements

1. **Linear Projections for Q, K, V**
   - Project input `q`, `k`, `v` into `d_model` dimensions using separate linear layers.

2. **Scaled Dot-Product Attention**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_model)`
   - Apply an optional `mask` before softmax.
   - Use the scores to weight `V`.

3. **Output Projection**
   - Apply a final linear projection to the attended output: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch's Reference**
   - Test your output against `torch.nn.MultiheadAttention` with `num_heads=1` using the same weights.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ No splitting into multiple heads — this is single-head attention.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention(num_heads=1)` output when weights are aligned.

---

<details>
  <summary>💡 Hint</summary>

  - Unlike multi-head attention, there is no need to `.view()` or `.transpose()` for head splitting.
  - Softmax should be applied over the **last dimension** (attention scores across the key sequence).
  - This is equivalent to multi-head attention with `num_heads=1`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F




def single_head_attention(q, k, v, d_model, mask=None):
    """
    Implements single-head attention.
    
    Args:
        q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
        k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
        v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
        d_model (int): Embedding dimension
        mask (Tensor, optional): Masking tensor for attention
        
    Returns:
        Tensor: Attention output of shape (batch_size, seq_len, d_model)
    """
    batch_size, seq_len, _ = q.shape
    
    # Linear projections for Q, K, V
    Q_w = nn.Linear(d_model, d_model, bias=False).to(device)
    K_w = nn.Linear(d_model, d_model, bias=False).to(device)
    V_w = nn.Linear(d_model, d_model, bias=False).to(device)
    W_out = nn.Linear(d_model, d_model, bias=False).to(device)

    Q = Q_w(q)  # (batch_size, seq_len, d_model)
    K = K_w(k)
    V = V_w(v)

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_model ** 0.5)  # (batch_size, seq_len, seq_len)

    # Mask check
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)  # (batch_size, seq_len, seq_len)

    output = torch.matmul(attn_weights, V)  # (batch_size, seq_len, d_model)

    return W_out(output)

In [ ]:
# Testing on data & compare
output_custom = single_head_attention(q, k, v, d_model)
print(output_custom)

multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=1, bias=False, batch_first=True)
output, _ = multihead_attn(q, k, v)
print(output)

assert torch.allclose(output_custom, output, atol=1e-08, rtol=1e-05)  # Check if they are close enough.


---

# 03 · Self-Attention

*Source: `torch/transformer/03-Self-Attention/self-attention.ipynb`*

---


# Implement Self-Attention from Scratch
### Problem Statement
Self-Attention is the mechanism where a sequence **attends to itself** — meaning Q, K, and V all come from the **same input**. This allows each token in a sequence to gather context from every other token.

Self-Attention is the core operation inside every Transformer encoder layer (e.g., BERT) and decoder layer (e.g., GPT), and understanding it is essential before moving to multi-head or cross-attention.

Your goal is to implement Self-Attention from scratch using PyTorch — projecting a single input `x` into Q, K, V, computing scaled dot-product attention, and producing the output.

---

### Requirements

1. **Linear Projections for Q, K, V from the same input**
   - Given a single input tensor `x`, project it into Q, K, and V using separate linear layers.

2. **Scaled Dot-Product Attention**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_model)`
   - Apply an optional `mask` before softmax (useful for causal/autoregressive masking).
   - Use the scores to weight `V`.

3. **Output Projection**
   - Apply a final linear projection to the attended output: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch's Reference**
   - Test your output against `torch.nn.MultiheadAttention` with `num_heads=1` using the same input for Q, K, V.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Q, K, V must all be derived from the **same input** `x`.
- ✅ Support optional masking (e.g., causal mask for autoregressive models).
- ✅ Must match `torch.nn.MultiheadAttention(num_heads=1)` output when Q=K=V=x.

---

<details>
  <summary>💡 Hint</summary>

  - The key difference from general attention: `q = k = v = x`. The same input is used for all three.
  - Softmax should be applied over the **last dimension** (attention scores across the sequence).
  - For causal masking, use `torch.tril()` to create a lower-triangular mask.
  - This is equivalent to calling `MultiheadAttention(x, x, x)` with `num_heads=1`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8

x = torch.rand(batch_size, seq_len, d_model)
print(x.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F




def self_attention(x, d_model, mask=None):
    """
    Implements self-attention (Q, K, V all derived from the same input).
    
    Args:
        x (Tensor): Input tensor of shape (batch_size, seq_len, d_model)
        d_model (int): Embedding dimension
        mask (Tensor, optional): Masking tensor for attention
        
    Returns:
        Tensor: Self-attention output of shape (batch_size, seq_len, d_model)
    """
    batch_size, seq_len, _ = x.shape
    
    # Linear projections for Q, K, V (all from the same input x)
    Q_w = nn.Linear(d_model, d_model, bias=False).to(device)
    K_w = nn.Linear(d_model, d_model, bias=False).to(device)
    V_w = nn.Linear(d_model, d_model, bias=False).to(device)
    W_out = nn.Linear(d_model, d_model, bias=False).to(device)

    Q = Q_w(x)  # (batch_size, seq_len, d_model)
    K = K_w(x)
    V = V_w(x)

    # Scaled dot-product attention
    scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_model ** 0.5)  # (batch_size, seq_len, seq_len)

    # Mask check
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)  # (batch_size, seq_len, seq_len)

    output = torch.matmul(attn_weights, V)  # (batch_size, seq_len, d_model)

    return W_out(output)

In [ ]:
# Testing on data & compare (without mask)
output_custom = self_attention(x, d_model)
print("Custom output:", output_custom.shape)
print(output_custom)

multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=1, bias=False, batch_first=True)
output, _ = multihead_attn(x, x, x)  # Self-attention: Q=K=V=x
print("\nPyTorch output:", output.shape)
print(output)

assert torch.allclose(output_custom, output, atol=1e-08, rtol=1e-05)  # Check if they are close enough.

In [ ]:
# Testing with causal mask
causal_mask = torch.tril(torch.ones(seq_len, seq_len))  # Lower-triangular mask
print("Causal mask:")
print(causal_mask)

output_masked = self_attention(x, d_model, mask=causal_mask)
print("\nMasked output shape:", output_masked.shape)
print(output_masked)


---

# 04 · Multi-Head Attention (MHA)

*Source: `torch/transformer/04-Multi-Head-Attention/multi-head-attention.ipynb`*

---


# Implement Attention from Scratch
### Problem Statement
Multi-Head Attention (MHA) is the bread-and-butter of the Transformer architecture. It enables the model to **jointly attend** to information from different representation subspaces at different positions.

Your goal is to implement MHA from scratch using PyTorch, simulating exactly what `torch.nn.MultiheadAttention` does — projecting Q, K, V for each head, computing attention weights, applying them to V, and concatenating the outputs across all heads.

---

### Requirements

1. **Linear Projections for Q, K, V**
   - Project input `q`, `k`, `v` into a total of `d_model` dimensions.
   - Split them into `num_heads` of `d_head = d_model // num_heads` each.

2. **Scaled Dot-Product Attention per Head**
   - Compute attention scores:  
     `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply an optional `mask` before softmax.
   - Use the scores to weight `V`.

3. **Combine the Heads**
   - Concatenate the outputs of all heads.
   - Apply a final linear projection to restore the shape: `(batch_size, seq_len, d_model)`.

4. **Validate Against PyTorch’s Reference**
   - Test your output against `torch.nn.MultiheadAttention` using the same input tensors.
   - Check for numerical closeness using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Make sure all tensors are reshaped properly when splitting and combining heads.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when heads and shape are aligned.

---

<details>
  <summary>💡 Hint</summary>

  - Use `.view()` and `.transpose()` to shape Q, K, V to `(batch_size, num_heads, seq_len, d_head)`.
  - Softmax should be applied over the **last dimension** (attention scores across sequence).
  - Use `.contiguous().view()` to flatten the multi-head outputs back into `(batch_size, seq_len, d_model)`.
  - Match PyTorch’s behavior using the same projections and batch-first format.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    Implements multi-head attention.
    """
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads  # Head size dimension

        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: Multi-head attention output of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = q.shape

        Q = self.Q_w(q)  # (batch_size, seq_len, d_model)
        K = self.K_w(k)
        V = self.V_w(v)

        # Reshape to (batch_size, num_heads, seq_len, d_head)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        # Mask check
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)

        output = torch.matmul(attn_weights, V)  # (batch_size, num_heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_out(output)

## 📚 Understanding the Head Combination Step

The most confusing line in multi-head attention is often:
```python
output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
```

Let's break down what each operation does:

### Starting Point
Before this line, `output` has shape: `(batch_size, num_heads, seq_len, d_head)`

For example: `(3, 2, 4, 4)` means:
- 3 samples in the batch
- 2 attention heads  
- 4 tokens in sequence
- 4 dimensions per head (where `d_model=8`, so `d_head=8/2=4`)

---

### Step 1: `.transpose(1, 2)` — Swap dimensions

Swaps dimensions 1 and 2 (num_heads ↔ seq_len):

```
Before: (batch_size, num_heads, seq_len, d_head)  →  (3, 2, 4, 4)
After:  (batch_size, seq_len, num_heads, d_head)  →  (3, 4, 2, 4)
```

**Why?** We need `seq_len` and `num_heads` next to each other so we can merge them.

---

### Step 2: `.contiguous()` — Fix memory layout

Makes the tensor's memory layout contiguous in RAM.

**Why?** `.transpose()` doesn't actually move data in memory—it just changes how PyTorch *views* it. Before we can use `.view()` to reshape, the memory must be laid out sequentially. `.contiguous()` reorganizes the data if needed.

---

### Step 3: `.view(batch_size, seq_len, d_model)` — Merge heads

Reshapes by concatenating the last two dimensions:

```
Before: (batch_size, seq_len, num_heads, d_head)  →  (3, 4, 2, 4)
After:  (batch_size, seq_len, d_model)            →  (3, 4, 8)
```

**What happens?** `num_heads × d_head = 2 × 4 = 8 = d_model` ✅

All heads are **concatenated** together for each token!

---

### Visual Example

```python
# Multi-head output: (batch=1, heads=2, seq=3, d_head=4)
[[[
    [[a1, a2, a3, a4],    # Head 0, Token 0
     [b1, b2, b3, b4],    # Head 0, Token 1  
     [c1, c2, c3, c4]],   # Head 0, Token 2
    
    [[d1, d2, d3, d4],    # Head 1, Token 0
     [e1, e2, e3, e4],    # Head 1, Token 1
     [f1, f2, f3, f4]]    # Head 1, Token 2
]]]

# After transpose(1,2): (batch=1, seq=3, heads=2, d_head=4)
[[[
    [[a1, a2, a3, a4], [d1, d2, d3, d4]],  # Token 0: [Head0, Head1]
    [[b1, b2, b3, b4], [e1, e2, e3, e4]],  # Token 1: [Head0, Head1]
    [[c1, c2, c3, c4], [f1, f2, f3, f4]]   # Token 2: [Head0, Head1]
]]]

# After view(1, 3, 8): (batch=1, seq=3, d_model=8)
[[[
    [a1, a2, a3, a4, d1, d2, d3, d4],  # Token 0: heads concatenated!
    [b1, b2, b3, b4, e1, e2, e3, e4],  # Token 1: heads concatenated!
    [c1, c2, c3, c4, f1, f2, f3, f4]   # Token 2: heads concatenated!
]]]
```

---

### Summary Table

| Operation | Input Shape | Output Shape | Purpose |
|-----------|-------------|--------------|---------|
| `.transpose(1,2)` | `(B, H, S, D)` | `(B, S, H, D)` | Move seq_len next to num_heads |
| `.contiguous()` | `(B, S, H, D)` | `(B, S, H, D)` | Fix memory layout for `.view()` |
| `.view(B, S, H×D)` | `(B, S, H, D)` | `(B, S, d_model)` | Concatenate all heads |

Where: `B=batch`, `H=num_heads`, `S=seq_len`, `D=d_head`, `H×D=d_model`

In [ ]:
# Testing on data & compare
# Create PyTorch's MultiheadAttention
multihead_attn = torch.nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True)

# Create our custom implementation
custom_attn = MultiHeadAttention(num_heads, d_model)

# Copy weights from PyTorch's implementation to ours for fair comparison
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

# Now compare outputs
output_custom = custom_attn(q, k, v)
print("Custom output:")
print(output_custom)

output, _ = multihead_attn(q, k, v)
print("\nPyTorch output:")
print(output)

assert torch.allclose(output_custom, output, atol=1e-06, rtol=1e-05)  # Check if they are close enough.
print("\n✅ Outputs match! Implementation is correct.")


---

# 05 · Multi-Query Attention (MQA)

*Source: `torch/transformer/05-Multi-Query-Attention/multi-query-attention.ipynb`*

---


# Implement Multi-Query Attention from Scratch
### Problem Statement
Multi-Query Attention (MQA), introduced in [Shazeer 2019](https://arxiv.org/abs/1911.02150), is a memory-efficient variant of Multi-Head Attention. Instead of having separate K and V projections per head, MQA uses a **single shared K/V head** across all query heads.

This dramatically reduces the KV cache size during autoregressive inference (from `num_heads × d_head` to just `1 × d_head` per token), making it much faster for long-sequence generation.

Your goal is to implement MQA from scratch using PyTorch and validate it against standard Multi-Head Attention in the degenerate case where `num_heads = 1`.

---

### Requirements

1. **Linear Projections**
   - Q projection: `d_model → d_model` (all query heads)
   - K projection: `d_model → d_head` (single shared head)
   - V projection: `d_model → d_head` (single shared head)
   - Output projection: `d_model → d_model`

2. **Expand K/V to Match Query Heads**
   - Use `.expand()` to broadcast the single K/V head across all query heads.

3. **Scaled Dot-Product Attention**
   - Compute: `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply optional mask before softmax.
   - Weight V with attention weights.

4. **Combine Heads**
   - Concatenate all query head outputs.
   - Apply final linear projection.

5. **Validate Against MHA**
   - When `num_heads = 1`, MQA is identical to single-head MHA.
   - Copy weights and verify outputs match with `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ K and V must be projected to a single head (`d_head`), not `d_model`.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when `num_heads = 1`.

---

<details>
  <summary>💡 Hint</summary>

  - Q shape after projection and reshape: `(batch, num_heads, seq_len, d_head)`
  - K/V shape after projection and reshape: `(batch, 1, seq_len, d_head)`
  - Use `.expand(-1, num_heads, -1, -1)` to broadcast K/V to all heads without copying memory.
  - When `num_heads = 1`, the Q projection is `d_model → d_model` which equals `d_model → 1 * d_head = d_head = d_model`, so all projections have the same shape — identical to single-head MHA.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class MultiQueryAttention(nn.Module):
    """
    Implements Multi-Query Attention (MQA).
    All query heads share a single key/value head.
    """
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        # Q projects to all heads, K/V project to a single shared head
        self.Q_w = nn.Linear(d_model, d_model, bias=False)            # d_model -> num_heads * d_head
        self.K_w = nn.Linear(d_model, self.d_head, bias=False)        # d_model -> 1 * d_head
        self.V_w = nn.Linear(d_model, self.d_head, bias=False)        # d_model -> 1 * d_head
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: MQA output of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = q.shape

        # Project Q to all heads, K/V to single head
        Q = self.Q_w(q)  # (batch_size, seq_len, d_model)
        K = self.K_w(k)  # (batch_size, seq_len, d_head)
        V = self.V_w(v)  # (batch_size, seq_len, d_head)

        # Reshape Q to (batch, num_heads, seq_len, d_head)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Reshape K, V to (batch, 1, seq_len, d_head)
        K = K.view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, 1, self.d_head).transpose(1, 2)

        # Expand K, V to match num_heads (no memory copy — just broadcasting)
        K = K.expand(-1, self.num_heads, -1, -1)  # (batch, num_heads, seq_len, d_head)
        V = V.expand(-1, self.num_heads, -1, -1)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_out(output)

## 📚 MQA vs MHA: What Changed?

| | MHA | MQA |
|---|---|---|
| Q projection | `d_model → d_model` | `d_model → d_model` (same) |
| K projection | `d_model → d_model` | `d_model → d_head` |
| V projection | `d_model → d_model` | `d_model → d_head` |
| K/V shape | `(B, num_heads, S, d_head)` | `(B, 1, S, d_head)` → expanded |
| KV cache per token | `num_heads × d_head × 2` | `d_head × 2` |
| Parameter count | `3 × d_model²` + output | `d_model² + 2 × d_model × d_head` + output |

### Why does this work?

The key insight is that **different query heads can still learn different attention patterns** even though they attend over the same K/V representations. The Q projection provides all the head-specific specialization, while K/V provide a shared "memory" that all heads read from.

### The expand trick

```python
K = K.expand(-1, self.num_heads, -1, -1)
```

`.expand()` doesn't copy data — it creates a view where the same memory is referenced by all heads. This is both memory-efficient and semantically correct: all heads truly share the same K/V.

Compare with `.repeat()` which actually copies the data:
```python
# expand: no copy, just a strided view (preferred)
K.expand(-1, num_heads, -1, -1)  
# repeat: copies data num_heads times (wastes memory)
K.repeat(1, num_heads, 1, 1)
```

In [ ]:
# Validate: when num_heads=1, MQA == single-head MHA
# (because there's only 1 query head and 1 KV head — identical)
num_heads_test = 1

multihead_attn = torch.nn.MultiheadAttention(
    embed_dim=d_model, num_heads=num_heads_test, bias=False, batch_first=True
)

custom_attn = MultiQueryAttention(num_heads_test, d_model)

# Copy weights: when num_heads=1, Q/K/V projections are all d_model -> d_model
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

output_custom = custom_attn(q, k, v)
output_ref, _ = multihead_attn(q, k, v)

print("Custom MQA output:")
print(output_custom)
print("\nPyTorch MHA output (num_heads=1):")
print(output_ref)

assert torch.allclose(output_custom, output_ref, atol=1e-06, rtol=1e-05)
print("\n✅ Outputs match! MQA implementation is correct.")

In [ ]:
# Bonus: verify KV cache savings
# MHA KV cache per token: num_heads * d_head * 2 (K and V)
# MQA KV cache per token: 1 * d_head * 2 (K and V)
print(f"MHA KV cache per token: {num_heads * (d_model // num_heads) * 2} values")
print(f"MQA KV cache per token: {1 * (d_model // num_heads) * 2} values")
print(f"Memory reduction: {num_heads}x")


---

# 06 · Grouped-Query Attention (GQA)

*Source: `torch/transformer/06-Grouped-Query-Attention/grouped-query-attention.ipynb`*

---


# Implement Grouped Query Attention from Scratch
### Problem Statement
Grouped Query Attention (GQA), introduced in [Ainslie et al. 2023](https://arxiv.org/abs/2305.13245), is a generalization that sits between Multi-Head Attention (MHA) and Multi-Query Attention (MQA).

- **MHA**: each query head has its own K/V head (`num_kv_heads == num_heads`)
- **MQA**: all query heads share a single K/V head (`num_kv_heads == 1`)
- **GQA**: query heads are divided into groups, each group shares one K/V head (`1 < num_kv_heads < num_heads`)

GQA strikes a balance — it reduces KV cache size (like MQA) while preserving more representational capacity (like MHA). It is used in Llama 2 70B, Llama 3, Mistral, and many modern LLMs.

Your goal is to implement GQA from scratch and validate it against `torch.nn.MultiheadAttention` in the degenerate case where `num_kv_heads == num_heads` (which is identical to MHA).

---

### Requirements

1. **Linear Projections**
   - Q projection: `d_model → d_model` (all query heads)
   - K projection: `d_model → num_kv_heads * d_head` (grouped KV heads)
   - V projection: `d_model → num_kv_heads * d_head` (grouped KV heads)
   - Output projection: `d_model → d_model`

2. **Expand K/V to Match Query Heads**
   - Use `repeat_interleave()` to duplicate each KV group to match the query heads in that group.

3. **Scaled Dot-Product Attention**
   - Compute: `scores = Q @ Kᵀ / sqrt(d_head)`
   - Apply optional mask before softmax.
   - Weight V with attention weights.

4. **Combine Heads and Output**
   - Concatenate all heads and apply final linear projection.

5. **Validate Against MHA**
   - When `num_kv_heads == num_heads`, GQA is identical to MHA.
   - Copy weights and verify outputs match with `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ `num_heads` must be divisible by `num_kv_heads`.
- ✅ Support optional masking.
- ✅ Must match `torch.nn.MultiheadAttention` output when `num_kv_heads == num_heads`.

---

<details>
  <summary>💡 Hint</summary>

  - Q shape after reshape: `(batch, num_heads, seq_len, d_head)`
  - K/V shape after reshape: `(batch, num_kv_heads, seq_len, d_head)`
  - `repeat_factor = num_heads // num_kv_heads`
  - Use `K.repeat_interleave(repeat_factor, dim=1)` to expand K/V from `num_kv_heads` to `num_heads`.
  - When `num_kv_heads == num_heads`, `repeat_factor = 1` so no duplication happens — identical to MHA.
  - When `num_kv_heads == 1`, this becomes MQA.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Implements Grouped Query Attention (GQA).
    Query heads are divided into groups, each sharing one K/V head.
    """
    def __init__(self, num_heads, num_kv_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0
        assert num_heads % num_kv_heads == 0

        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads
        self.repeat_factor = num_heads // num_kv_heads

        # Q projects to all heads, K/V project to num_kv_heads only
        self.Q_w = nn.Linear(d_model, d_model, bias=False)                          # d_model -> num_heads * d_head
        self.K_w = nn.Linear(d_model, self.num_kv_heads * self.d_head, bias=False)  # d_model -> num_kv_heads * d_head
        self.V_w = nn.Linear(d_model, self.num_kv_heads * self.d_head, bias=False)  # d_model -> num_kv_heads * d_head
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, q, k, v, mask=None):
        """
        Args:
            q (Tensor): Query tensor of shape (batch_size, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch_size, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Masking tensor for attention

        Returns:
            Tensor: GQA output of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = q.shape

        # Project Q to all heads, K/V to num_kv_heads
        Q = self.Q_w(q)  # (batch_size, seq_len, d_model)
        K = self.K_w(k)  # (batch_size, seq_len, num_kv_heads * d_head)
        V = self.V_w(v)  # (batch_size, seq_len, num_kv_heads * d_head)

        # Reshape Q to (batch, num_heads, seq_len, d_head)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Reshape K, V to (batch, num_kv_heads, seq_len, d_head)
        K = K.view(batch_size, seq_len, self.num_kv_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_kv_heads, self.d_head).transpose(1, 2)

        # Expand K, V from num_kv_heads -> num_heads by repeating each group
        K = K.repeat_interleave(self.repeat_factor, dim=1)  # (batch, num_heads, seq_len, d_head)
        V = V.repeat_interleave(self.repeat_factor, dim=1)

        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)  # (batch, num_heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_out(output)

## 📚 The GQA Spectrum

GQA generalizes both MHA and MQA through the `num_kv_heads` parameter:

```
num_kv_heads = num_heads  →  MHA (each head has its own K/V)
num_kv_heads = 1          →  MQA (all heads share one K/V)
1 < num_kv_heads < H      →  GQA (groups of heads share K/V)
```

### Example: 8 query heads, 2 KV heads

```
Query heads:  [Q0, Q1, Q2, Q3] [Q4, Q5, Q6, Q7]
                    ↓                   ↓
KV heads:         (K0, V0)           (K1, V1)
```

Heads Q0–Q3 share K0/V0, heads Q4–Q7 share K1/V1.

### repeat_interleave vs expand

- **MQA** uses `.expand()` — broadcasts a single head to all positions (no data copy)
- **GQA** uses `.repeat_interleave()` — duplicates each group's K/V to fill its query heads

```python
# GQA: K shape (batch, 2, seq, d_head) -> (batch, 8, seq, d_head)
# repeat_interleave(4, dim=1) gives: [K0,K0,K0,K0, K1,K1,K1,K1]
K = K.repeat_interleave(repeat_factor, dim=1)
```

### KV Cache Savings

| Variant | KV cache per token | Relative to MHA |
|---------|-------------------|------------------|
| MHA (H=8, kv=8) | `8 × d_head × 2` | 1× |
| GQA (H=8, kv=2) | `2 × d_head × 2` | 4× smaller |
| MQA (H=8, kv=1) | `1 × d_head × 2` | 8× smaller |

### Real-world usage

| Model | num_heads | num_kv_heads | Variant |
|-------|-----------|-------------|--------|
| GPT-3 | 96 | 96 | MHA |
| PaLM | 16 | 1 | MQA |
| Llama 2 70B | 64 | 8 | GQA |
| Llama 3 8B | 32 | 8 | GQA |
| Mistral 7B | 32 | 8 | GQA |

In [ ]:
# Validate: when num_kv_heads == num_heads, GQA == MHA
multihead_attn = torch.nn.MultiheadAttention(
    embed_dim=d_model, num_heads=num_heads, bias=False, batch_first=True
)

custom_attn = GroupedQueryAttention(num_heads, num_kv_heads=num_heads, d_model=d_model)

# Copy weights: when num_kv_heads == num_heads, K/V projections are d_model -> d_model (same as MHA)
custom_attn.Q_w.weight.data = multihead_attn.in_proj_weight[:d_model, :].clone()
custom_attn.K_w.weight.data = multihead_attn.in_proj_weight[d_model:2*d_model, :].clone()
custom_attn.V_w.weight.data = multihead_attn.in_proj_weight[2*d_model:, :].clone()
custom_attn.W_out.weight.data = multihead_attn.out_proj.weight.data.clone()

output_custom = custom_attn(q, k, v)
output_ref, _ = multihead_attn(q, k, v)

print("Custom GQA output:")
print(output_custom)
print("\nPyTorch MHA output:")
print(output_ref)

assert torch.allclose(output_custom, output_ref, atol=1e-06, rtol=1e-05)
print("\n✅ Outputs match! GQA implementation is correct.")

In [ ]:
# Bonus: KV cache comparison across all three variants
d_head = d_model // num_heads
print(f"d_model={d_model}, num_heads={num_heads}, d_head={d_head}")
print(f"")
print(f"MHA (num_kv_heads={num_heads}):  KV cache = {num_heads * d_head * 2} values/token")
print(f"GQA (num_kv_heads=1):            KV cache = {1 * d_head * 2} values/token  (same as MQA)")
print(f"MQA (num_kv_heads=1):            KV cache = {1 * d_head * 2} values/token")
print(f"")
print(f"GQA is a spectrum: num_kv_heads=1 → MQA, num_kv_heads=num_heads → MHA")


---

# 07 · LayerNorm

*Source: `torch/transformer/07-Layer-Norm/layer-norm.ipynb`*

---


# Implement Layer Normalization from Scratch
### Problem Statement
Layer Normalization (LayerNorm), introduced in [Ba et al. 2016](https://arxiv.org/abs/1607.06450), normalizes activations across the **feature dimension** for each individual sample. Unlike Batch Normalization (which normalizes across the batch), LayerNorm is independent of batch size and works naturally for sequence models.

LayerNorm is used in the original Transformer (Vaswani et al. 2017), GPT-2, GPT-3, and BERT. Modern LLMs have largely replaced it with RMSNorm, but understanding LayerNorm is foundational.

Your goal is to implement LayerNorm from scratch and validate against `torch.nn.LayerNorm`.

---

### Requirements

1. **Compute Mean and Variance**
   - Mean: `mu = mean(x)` over the last dimension
   - Variance: `sigma² = var(x)` over the last dimension (unbiased=False)

2. **Normalize**
   - `x_norm = (x - mu) / sqrt(sigma² + eps)`

3. **Affine Transform**
   - Learnable `gamma` (scale, init ones) and `beta` (bias, init zeros)
   - `output = gamma * x_norm + beta`

4. **Validate Against PyTorch**
   - Compare with `torch.nn.LayerNorm` using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Normalize over the last dimension.
- ✅ Must match `torch.nn.LayerNorm` output.

---

<details>
  <summary>💡 Hint</summary>

  - Use `x.mean(dim=-1, keepdim=True)` and `x.var(dim=-1, keepdim=True, unbiased=False)`.
  - PyTorch's `LayerNorm` uses **population variance** (unbiased=False), not sample variance.
  - `gamma = nn.Parameter(torch.ones(dim))`, `beta = nn.Parameter(torch.zeros(dim))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class LayerNorm(nn.Module):
    """
    Implements Layer Normalization.
    """
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta = nn.Parameter(torch.zeros(dim))   # shift

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (Tensor): Input tensor of shape (..., dim)

        Returns:
            Tensor: Normalized tensor of same shape
        """
        # Compute mean and variance over the last dimension
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        # Scale and shift
        return self.gamma * x_norm + self.beta

## 📚 LayerNorm: Key Concepts

### Why unbiased=False?

PyTorch's `nn.LayerNorm` uses **population variance** (divides by N, not N-1). This is because:
- We're normalizing a fixed set of features, not estimating a population parameter
- The feature dimension is the entire "population" for each sample

```python
# Population variance (unbiased=False): sum((x - mean)²) / N
# Sample variance (unbiased=True):      sum((x - mean)²) / (N-1)
x.var(dim=-1, unbiased=False)  # This matches nn.LayerNorm
```

### BatchNorm vs LayerNorm

```
Input shape: (batch_size, seq_len, dim)

BatchNorm:  normalizes across batch dimension     → stats shape: (dim,)
LayerNorm:  normalizes across feature dimension   → stats shape: (batch, seq_len, 1)
```

| Property | BatchNorm | LayerNorm |
|----------|-----------|----------|
| Normalizes over | Batch dim | Feature dim |
| Batch-size dependent? | Yes | No |
| Running stats at inference? | Yes | No |
| Common in | CNNs | Transformers |

### Pre-Norm vs Post-Norm

```
Post-Norm (original Transformer):     x + Sublayer(LayerNorm(x))  ← NO
Actually:                              LayerNorm(x + Sublayer(x))  ← YES

Pre-Norm (GPT-2, modern LLMs):        x + Sublayer(Norm(x))
```

Pre-Norm is more stable for training deep models because the residual path is unobstructed.

In [ ]:
# Validate against PyTorch's LayerNorm
torch.manual_seed(42)
x = torch.randn(3, 4, 8)  # (batch, seq_len, dim)

custom_ln = LayerNorm(dim=8)
pytorch_ln = nn.LayerNorm(8)

# Copy weights for fair comparison
custom_ln.gamma.data = pytorch_ln.weight.data.clone()
custom_ln.beta.data = pytorch_ln.bias.data.clone()

output_custom = custom_ln(x)
output_ref = pytorch_ln(x)

print("Custom output:")
print(output_custom[0, 0, :])
print("\nPyTorch output:")
print(output_ref[0, 0, :])

assert torch.allclose(output_custom, output_ref, atol=1e-6)
print("\n✅ Outputs match! LayerNorm implementation is correct.")


---

# 08 · RMSNorm

*Source: `torch/transformer/08-RMS-Norm/rms-norm.ipynb`*

---


# Implement RMS Normalization from Scratch
### Problem Statement
RMS Normalization (RMSNorm), introduced in [Zhang & Sennrich 2019](https://arxiv.org/abs/1910.07467), is a simplified alternative to Layer Normalization. Instead of computing both mean and variance, RMSNorm only computes the **root mean square** of the activations.

This makes it ~10-15% faster than LayerNorm while achieving comparable performance. RMSNorm is the default normalization in modern LLMs including Llama, Llama 2/3, Mistral, and Gemma.

---

### Requirements

1. **RMS Computation**
   - Compute: `rms = sqrt(mean(x²) + eps)`
   - Normalize: `x_norm = x / rms`

2. **Learnable Scale Parameter**
   - Apply a learnable `gamma` (initialized to ones): `output = x_norm * gamma`
   - Note: RMSNorm has **no bias** and **no mean centering** — this is the key difference from LayerNorm.

3. **Validate**
   - Output shape must match input shape.
   - Compare behavior with a manual computation.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Support arbitrary input shapes `(..., dim)` — normalize over the last dimension.
- ✅ Include `eps` for numerical stability.

---

<details>
  <summary>💡 Hint</summary>

  - `x.pow(2).mean(dim=-1, keepdim=True)` gives you the mean of squared values.
  - `torch.sqrt(...)` to get the RMS.
  - The scale parameter `gamma` is `nn.Parameter(torch.ones(dim))`.
  - Unlike LayerNorm, there is no mean subtraction step.

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class RMSNorm(nn.Module):
    """
    Implements Root Mean Square Layer Normalization.
    """
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))  # gamma

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (Tensor): Input tensor of shape (..., dim)

        Returns:
            Tensor: Normalized tensor of same shape
        """
        # Compute RMS: sqrt(mean(x^2) + eps)
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

        # Normalize and scale
        return (x / rms) * self.scale

## 📚 RMSNorm vs LayerNorm

### LayerNorm formula
```
LayerNorm(x) = gamma * (x - mean(x)) / sqrt(var(x) + eps) + beta
```
- Computes **mean** and **variance**
- Has both **gamma** (scale) and **beta** (bias)
- Re-centers the distribution (subtracts mean)

### RMSNorm formula
```
RMSNorm(x) = gamma * x / sqrt(mean(x²) + eps)
```
- Only computes **root mean square** (no mean subtraction)
- Has only **gamma** (no bias)
- Simpler and faster — skips the mean computation and bias addition

### Why does removing the mean centering work?

The authors showed that the **re-scaling invariance** (not the re-centering) is the key property that makes normalization effective for training stability. Removing mean centering:
- Reduces computation by ~10-15%
- Has negligible impact on model quality
- Is particularly beneficial for large models where normalization is called many times

### Where it's used

| Model | Normalization | Position |
|-------|--------------|----------|
| GPT-2/3 | LayerNorm | Post-attention, post-FFN |
| Llama 1/2/3 | RMSNorm | **Pre**-attention, **Pre**-FFN |
| Mistral | RMSNorm | Pre-attention, Pre-FFN |
| Gemma | RMSNorm | Pre-attention, Pre-FFN |

In [ ]:
# Test
torch.manual_seed(42)
x = torch.randn(3, 4, 8)  # (batch, seq_len, dim)
rmsnorm = RMSNorm(dim=8)
out = rmsnorm(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == x.shape, "Output shape mismatch"

# Manual verification
rms_manual = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + 1e-8)
expected = x / rms_manual  # gamma is all ones initially
assert torch.allclose(out, expected, atol=1e-6)
print("\n✅ RMSNorm implementation is correct.")


---

# 09 · Residual Connection

*Source: `torch/transformer/09-Residual-Connection/residual-connection.ipynb`*

---


# Implement Residual (Skip) Connection from Scratch
### Problem Statement
Residual connections (skip connections), introduced in [He et al. 2015](https://arxiv.org/abs/1512.03385) for ResNets, are a critical component of the Transformer. The idea is simple but powerful: instead of learning `F(x)`, learn `F(x) + x`. This allows gradients to flow directly through the network, enabling training of very deep models.

In Transformers, residual connections wrap both the attention sublayer and the feed-forward sublayer. Combined with normalization, there are two common patterns:

- **Post-Norm** (original Transformer): `LayerNorm(x + Sublayer(x))`
- **Pre-Norm** (GPT-2, Llama, modern LLMs): `x + Sublayer(Norm(x))`

Your goal is to implement both patterns and understand why Pre-Norm is preferred for deep models.

---

### Requirements

1. **Post-Norm Residual Block**
   - `output = LayerNorm(x + sublayer(x))`
   - Normalization happens **after** the residual addition.

2. **Pre-Norm Residual Block**
   - `output = x + sublayer(Norm(x))`
   - Normalization happens **before** the sublayer, residual path is clean.

3. **Demonstrate Gradient Flow**
   - Show that residual connections preserve gradient magnitude through many layers.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ The sublayer can be any `nn.Module` (attention, FFN, etc.).
- ✅ Both Pre-Norm and Post-Norm must be implemented.

---

<details>
  <summary>💡 Hint</summary>

  - The residual connection is literally just `x + f(x)` — the key design choice is where normalization goes.
  - Pre-Norm leaves the residual stream "clean" — the identity path `x` has no transformations applied to it.
  - Optional dropout is often applied to `sublayer(x)` before adding back to `x`.

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class PostNormResidual(nn.Module):
    """
    Post-Norm residual connection: LayerNorm(x + Sublayer(x))
    Used in the original Transformer (Vaswani et al. 2017).
    """
    def __init__(self, d_model: int, sublayer: nn.Module, dropout: float = 0.1):
        super().__init__()
        self.sublayer = sublayer
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm(x + self.dropout(self.sublayer(x)))

In [ ]:
class PreNormResidual(nn.Module):
    """
    Pre-Norm residual connection: x + Sublayer(Norm(x))
    Used in GPT-2, Llama, and most modern LLMs.
    """
    def __init__(self, d_model: int, sublayer: nn.Module, dropout: float = 0.1):
        super().__init__()
        self.sublayer = sublayer
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.dropout(self.sublayer(self.norm(x)))

## 📚 Why Residual Connections Matter

### The Gradient Flow Problem

Without residual connections, gradients must pass through every transformation layer during backpropagation. In a deep network with layers `f1, f2, ..., fN`, the gradient at the input is:

```
∂L/∂x = ∂L/∂fN · ∂fN/∂fN-1 · ... · ∂f2/∂f1 · ∂f1/∂x
```

This chain of multiplications causes gradients to either **vanish** (if factors < 1) or **explode** (if factors > 1).

### How Residual Connections Help

With `y = x + f(x)`, the gradient becomes:

```
∂y/∂x = 1 + ∂f(x)/∂x
```

The **+1 term** guarantees the gradient is at least 1, preventing vanishing gradients. Through N layers:

```
∂L/∂x = ∂L/∂yN · (1 + ∂fN/∂x) · (1 + ∂fN-1/∂x) · ... · (1 + ∂f1/∂x)
```

Every term in the product is at least 1, so gradients can flow freely.

### Pre-Norm vs Post-Norm

```
Post-Norm: LayerNorm(x + Sublayer(x))
  - Norm is on the residual path → gradients must pass through norm
  - Harder to train for very deep models
  - May need learning rate warmup

Pre-Norm:  x + Sublayer(Norm(x))
  - Residual path is completely clean (just addition)
  - Gradients flow through identity path unobstructed
  - More stable training, especially for deep models
  - Used by GPT-2, GPT-3, Llama, Mistral, etc.
```

### The Residual Stream View

Modern interpretability research views the transformer as a **residual stream** — information flows through the identity path, and each attention/FFN layer reads from and writes to this stream:

```
x₀ ──────────────────────────────────────────────> x_final
    ↘ Norm → Attn ↗  ↘ Norm → FFN ↗  ↘ Norm → Attn ↗
       Layer 1            Layer 1          Layer 2
```

In [ ]:
# Test both patterns
torch.manual_seed(42)
d_model = 8
x = torch.randn(3, 4, d_model)

# Use a simple linear layer as the sublayer
sublayer = nn.Linear(d_model, d_model)

post_norm = PostNormResidual(d_model, sublayer, dropout=0.0)
pre_norm = PreNormResidual(d_model, sublayer, dropout=0.0)

out_post = post_norm(x)
out_pre = pre_norm(x)

print(f"Input shape:      {x.shape}")
print(f"Post-Norm output: {out_post.shape}")
print(f"Pre-Norm output:  {out_pre.shape}")
assert out_post.shape == x.shape
assert out_pre.shape == x.shape
print("\n✅ Both residual connection patterns work correctly.")

In [ ]:
# Bonus: Gradient flow comparison
# Stack many layers and check gradient magnitude at the input

def check_gradient_flow(num_layers, use_residual=True):
    """Stack num_layers linear layers and measure gradient at input."""
    d = 64
    x = torch.randn(1, d, requires_grad=True)
    out = x
    layers = [nn.Linear(d, d, bias=False) for _ in range(num_layers)]
    
    for layer in layers:
        if use_residual:
            out = out + layer(out)  # residual
        else:
            out = layer(out)         # no residual
    
    loss = out.sum()
    loss.backward()
    return x.grad.norm().item()

for n in [5, 20, 50]:
    grad_residual = check_gradient_flow(n, use_residual=True)
    grad_plain = check_gradient_flow(n, use_residual=False)
    print(f"Layers={n:2d} | With residual: {grad_residual:.4f} | Without: {grad_plain:.6f}")


---

# 10 · Feed-Forward Network

*Source: `torch/transformer/10-Feed-Forward-Network/feed-forward-network.ipynb`*

---


# Implement Feed-Forward Network from Scratch
### Problem Statement
The Feed-Forward Network (FFN) is the other main component in each Transformer layer (alongside attention). It's a simple **position-wise** MLP applied independently to each token.

The original Transformer uses a two-layer MLP with ReLU:
```
FFN(x) = W2 · ReLU(W1 · x + b1) + b2
```

Modern LLMs (Llama, Mistral, Gemma) use **SwiGLU**, a gated variant:
```
SwiGLU(x) = W2 · (Swish(W_gate · x) ⊙ (W_up · x))
```
where `Swish(x) = x · sigmoid(x)` and `⊙` is element-wise multiplication.

Your goal is to implement both variants from scratch.

---

### Requirements

1. **Standard FFN (ReLU)**
   - Two linear layers: `d_model → d_ff → d_model`
   - ReLU activation between them
   - Typically `d_ff = 4 * d_model`

2. **SwiGLU FFN**
   - Three linear layers: `W_gate`, `W_up` (both `d_model → d_ff`), `W_down` (`d_ff → d_model`)
   - Gate: `Swish(W_gate(x))` where `Swish(x) = x * sigmoid(x)` (same as `F.silu`)
   - Output: `W_down(Swish(W_gate(x)) * W_up(x))`

3. **Validate**
   - Output shape must be `(batch_size, seq_len, d_model)` for both.
   - Verify parameter counts match expectations.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Both variants must maintain the input/output dimension `d_model`.
- ✅ SwiGLU should use `F.silu` (Swish activation).

---

<details>
  <summary>💡 Hint</summary>

  - Standard FFN: `nn.Linear(d_model, d_ff)` → `F.relu()` → `nn.Linear(d_ff, d_model)`
  - SwiGLU has **3 weight matrices** instead of 2, but `d_ff` is often set to `(2/3) * 4 * d_model` to keep total params similar.
  - `F.silu(x)` is `x * torch.sigmoid(x)` — the Swish activation.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class FeedForward(nn.Module):
    """
    Standard Transformer FFN with ReLU activation.
    FFN(x) = W2 * ReLU(W1 * x + b1) + b2
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(self.dropout(F.relu(self.w1(x))))

In [ ]:
class SwiGLUFeedForward(nn.Module):
    """
    SwiGLU Feed-Forward Network used in Llama, Mistral, Gemma.
    SwiGLU(x) = W_down * (Swish(W_gate * x) * W_up * x)
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        # SwiGLU typically uses (2/3) * 4 * d_model for d_ff
        # to keep param count similar to standard FFN (3 matrices of d*d_ff vs 2 matrices of d*4d)
        d_ff = d_ff or int(2 / 3 * 4 * d_model)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Gate branch: apply Swish activation
        gate = F.silu(self.w_gate(x))  # Swish = x * sigmoid(x)
        # Up branch: linear projection
        up = self.w_up(x)
        # Element-wise product of gate and up, then project down
        return self.w_down(self.dropout(gate * up))

## 📚 FFN Variants Explained

### Standard FFN (Original Transformer, GPT-2/3, BERT)

```
x ──→ [W1: d→4d] ──→ ReLU ──→ [W2: 4d→d] ──→ output
```

- 2 weight matrices: `W1 (d × 4d)` + `W2 (4d × d)` = **8d²** params
- Simple and effective

### SwiGLU FFN (Llama, Mistral, PaLM, Gemma)

```
         ┌→ [W_gate: d→d'] → Swish ──┐
x ──────┤                              ⊙ ──→ [W_down: d'→d] → output
         └→ [W_up:   d→d'] ──────────┘
```

- 3 weight matrices: `W_gate (d × d')` + `W_up (d × d')` + `W_down (d' × d)` = **3dd'** params
- With `d' = (2/3) * 4d`: 3 × d × (8d/3) = **8d²** params (same as standard!)
- The gating mechanism allows the network to selectively pass information

### Why SwiGLU?

1. **Gating**: The element-wise product `Swish(gate) ⊙ up` lets the network learn to selectively activate dimensions
2. **Swish**: Smoother than ReLU, allows small negative values to pass through
3. **Empirically better**: PaLM paper showed SwiGLU consistently outperforms ReLU/GELU FFN

### Activation Functions Comparison

| Activation | Formula | Used in |
|-----------|---------|--------|
| ReLU | `max(0, x)` | Original Transformer |
| GELU | `x * Φ(x)` | BERT, GPT-2 |
| Swish/SiLU | `x * sigmoid(x)` | Llama (inside SwiGLU) |
| SwiGLU | `Swish(Wx) ⊙ Vx` | Llama, Mistral, PaLM |

In [ ]:
# Test both FFN variants
torch.manual_seed(42)
d_model = 64
x = torch.randn(3, 4, d_model)  # (batch, seq_len, d_model)

ffn = FeedForward(d_model, dropout=0.0)
swiglu = SwiGLUFeedForward(d_model, dropout=0.0)

out_ffn = ffn(x)
out_swiglu = swiglu(x)

print(f"Input shape:       {x.shape}")
print(f"FFN output shape:  {out_ffn.shape}")
print(f"SwiGLU output:     {out_swiglu.shape}")
assert out_ffn.shape == x.shape
assert out_swiglu.shape == x.shape

# Compare parameter counts
ffn_params = sum(p.numel() for p in ffn.parameters())
swiglu_params = sum(p.numel() for p in swiglu.parameters())
print(f"\nFFN params:    {ffn_params:,}")
print(f"SwiGLU params: {swiglu_params:,}")
print("\n✅ Both FFN variants work correctly.")

In [ ]:
# Bonus: Verify that F.silu matches manual Swish computation
x_test = torch.randn(5)
swish_manual = x_test * torch.sigmoid(x_test)
swish_pytorch = F.silu(x_test)
assert torch.allclose(swish_manual, swish_pytorch)
print("✅ F.silu == x * sigmoid(x) confirmed")


---

# 11 · Sinusoidal Positional Embedding

*Source: `torch/transformer/11-Sinusoidal-Positional-Embedding/sinusoidal-positional-embedding.ipynb`*

---


# Implement Attention from Scratch
###  Problem Statement
Transformers are order-agnostic — they see tokens like goldfish: no sense of sequence. To inject **position awareness** into the model, we use **Sinusoidal Positional Embeddings**, where each position in the sequence gets a unique deterministic vector. These vectors are computed using sine and cosine waves at different frequencies.

Your task is to implement the sinusoidal position encoding mechanism from scratch using PyTorch — no cheating with built-ins from `fairseq` or Hugging Face.

---

###  Requirements

1. **Define the Sinusoidal Embedding Class**
   - Implement a `SinusoidalPositionalEmbedding` class inheriting from `nn.Module`.
   - Initialize with `max_seq_len` and `d_model`.
   - Create a tensor `pe` of shape `(max_seq_len, d_model)` filled with sine and cosine encodings:
     - `sin(position * ω)` for even indices
     - `cos(position * ω)` for odd indices

2. **Register as Buffer**
   - Use `self.register_buffer("pe", pe)` to store `pe` without treating it as a trainable parameter.

3. **Generate Encodings**
   - On calling `forward(x)`, return the slice of positional encodings matching the sequence length of `x`.

4. **Test the Embeddings**
   - Initialize the embedding class with `max_seq_len = 100` and `d_model = 64`.
   - Pass a sequence of length 50 to verify the returned shape is `(1, 50, 64)`.

---

### Constraints

- ✅ Do not use Hugging Face, Fairseq, or built-in PyTorch modules for position encoding.
- ✅ Ensure the `pe` tensor is not a trainable parameter.
- ✅ Support any sequence length up to `max_seq_len`.
- ❌ Do not inject these embeddings directly into token embeddings yet — this is just the embedding module.

---

<details>
  <summary>💡 Hint</summary>

  - Use `torch.arange(0, max_seq_len).unsqueeze(1)` to create position indices.
  - Compute frequencies with `torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))`.
  - Alternate `sin` and `cos` values for even and odd embedding dimensions.
  - When returning the embedding in `forward`, use `.unsqueeze(0)` to broadcast over the batch dimension.

</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class SinusoidalPositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len: int, d_model: int):
        """
        Initializes the sinusoidal positional embedding.
        
        Args:
            max_seq_len (int): Maximum sequence length.
            d_model (int): Embedding dimension.
        """
        super().__init__()
        
        # Create a matrix of shape (max_seq_len, d_model)
        pe = torch.zeros(max_seq_len, d_model)
        
        # Position indices (0, 1, 2, ..., max_seq_len-1)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term using the exponential decay formula
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # Apply sin to even indices and cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Register as buffer (not a parameter, but saved in the model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        """
        Returns the positional embedding for a given input tensor.
        
        Args:
            x (Tensor): Input tensor of shape (batch_size, seq_len, d_model).
        
        Returns:
            Tensor: Positional embeddings of shape (batch_size, seq_len, d_model).
        """
        return self.pe[:x.shape[1], :].unsqueeze(0)  # Shape: (1, seq_len, d_model)

In [ ]:
# from fairseq.modules.sinusoidal_positional_embedding import SinusoidalPositionalEmbedding

max_seq_len = 100
d_model = 64

# Fairseq's implementation requires the number of embeddings (seq length) and embedding dim
# pos_emb = SinusoidalPositionalEmbedding(d_model, max_seq_len, padding_idx=None)

# Generate embeddings for a sequence of length 50
seq_len = 50
positions = torch.arange(seq_len).unsqueeze(0)  # Shape: (1, seq_len)
# positional_encoding = pos_emb(positions)  # Shape: (1, seq_len, d_model)

custom_pos_emb = SinusoidalPositionalEmbedding(d_model, max_seq_len)

positional_encoding_custom = custom_pos_emb(positions)

print(positional_encoding_custom.shape)  # (1, 50, 64)



---

# 12 · Rotary Positional Embedding (RoPE)

*Source: `torch/transformer/12-Rotary-Positional-Embedding/rotary-positional-embedding.ipynb`*

---


# Implement ROPE from Scratch
# 🔁 Rotary Positional Embeddings in PyTorch

### 🧠 Problem Statement
Transformers need a sense of **order**, but vanilla attention mechanisms are position-agnostic. Positional encodings help inject this order-awareness into the model. 

Your mission is to implement **Rotary Positional Embeddings (RoPE)** from scratch — a newer and slicker technique that rotates the query and key vectors instead of simply adding sine-cosine vectors. This method preserves attention efficiency while enabling better generalization for long sequences.

---

### ✅ Requirements

1. **Implement the Rotary Module**
   - Construct a `Rotary` class to compute sinusoidal frequencies.
   - Precompute and cache `cos` and `sin` values per sequence length.
   - Register these as buffers to keep them on the correct device.

2. **Define Rotation Helpers**
   - `rotate_half(x)` splits and rotates half the dimensions of a tensor.
   - `apply_rotary_pos_emb(q, k, cos, sin)` applies these rotations to Q and K.

3. **Simulate Usage**
   - Create synthetic tensors for Q, K, V.
   - Generate rotary embeddings using the custom `Rotary` module.
   - Apply rotary embeddings to Q and K.

4. **Verify Dimensions**
   - Final shapes should align with expected shapes for attention modules.
   - Confirm RoPE is applied before dot-product attention would normally occur.

---

### 📏 Constraints

- ✅ Use only PyTorch — no Fairseq or HuggingFace positional modules.
- ✅ Must support dynamic sequence lengths and cache embeddings per sequence.
- ✅ Should handle odd/even dimensional splits correctly.
- ❌ Do **not** manually plug in Fairseq’s `SinusoidalPositionalEmbedding`.

---

<details>
  <summary>💡 Hint</summary>
  - Use `torch.einsum("i,j->ij", t, inv_freq)` to compute frequency pairs.
  - Cache the cosine and sine values in the `Rotary` class using `self.register_buffer()`.
  - The `rotate_half(x)` function should split `x` into two halves and rotate them: `[-x2, x1]`.
  - Apply the rotary transformation using:  
    `(q * cos) + (rotate_half(q) * sin)`  
    and similarly for `k`.
  - Remember to broadcast `cos` and `sin` to match the shape of `q` and `k`.
</details>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math

In [ ]:
# Synthetic data
torch.manual_seed(42)
batch_size = 3
seq_len = 4
d_model = 8
num_heads = 2

q = torch.rand(batch_size, seq_len, d_model)
k = torch.rand(batch_size, seq_len, d_model)
v = torch.rand(batch_size, seq_len, d_model)
print(q.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
device = "cpu"

In [ ]:
class Rotary(torch.nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.seq_len_cached = None
        self.cos_cached = None
        self.sin_cached = None

    def forward(self, x, seq_dim=1):
        seq_len = x.shape[seq_dim]
        if seq_len != self.seq_len_cached:
            self.seq_len_cached = seq_len
            t = torch.arange(x.shape[seq_dim], device=x.device).type_as(self.inv_freq)
            freqs = torch.einsum("i,j->ij", t, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1).to(x.device)
            self.cos_cached = emb.cos()[:, None, None, :]
            self.sin_cached = emb.sin()[:, None, None, :]
        return self.cos_cached, self.sin_cached


# rotary pos emb helpers:

def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat(
        (-x2, x1), dim=x1.ndim - 1
    )  # dim=-1 triggers a bug in torch < 1.8.0


@torch.jit.script
def apply_rotary_pos_emb(q, k, cos, sin):
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

In [ ]:
# from fairseq.modules.sinusoidal_positional_embedding import SinusoidalPositionalEmbedding

max_seq_len = 100
d_model = 64

# Fairseq's implementation requires the number of embeddings (seq length) and embedding dim
# pos_emb = SinusoidalPositionalEmbedding(d_model, max_seq_len, padding_idx=None)

# Generate embeddings for a sequence of length 50
seq_len = 50
positions = torch.arange(seq_len).unsqueeze(0)  # Shape: (1, seq_len)
# positional_encoding = pos_emb(positions)  # Shape: (1, seq_len, d_model)

custom_pos_emb = Rotary(d_model, max_seq_len)

positional_encoding_custom = apply_rotary_pos_emb(positions)

print(positional_encoding_custom.shape)  # (1, 50, 64)



---

# 13 · KV Cache

*Source: `torch/transformer/13-KV-Cache/kv-cache.ipynb`*

---


# Implement KV-Cache from Scratch
### Problem Statement
During autoregressive text generation, the model generates one token at a time. Without caching, at step `t` we'd recompute attention over all `t` previous tokens — `O(t²)` total work for generating a sequence of length `T`.

The **KV-Cache** eliminates this redundancy: we cache the K and V projections of all previous tokens. At each new step, we only compute Q/K/V for the **new token** and append K/V to the cache.

This reduces per-step attention from `O(t × d)` to `O(d)` for the projection, making generation **much** faster.

Your goal is to implement an attention module with KV-Cache and show the speedup.

---

### Requirements

1. **CachedAttention Module**
   - On first call (prefill): compute full Q, K, V and initialize cache.
   - On subsequent calls (decode): compute Q, K, V for new token only, append K/V to cache.
   - Attention is always computed over the full cached K/V sequence.

2. **Cache Management**
   - Store K and V caches as tensors of shape `(batch, num_heads, cached_len, d_head)`.
   - Grow the cache by concatenating new K/V along the sequence dimension.

3. **Validate**
   - The cached incremental output at each step must match the non-cached full recomputation.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Cache must grow correctly across multiple decode steps.
- ✅ Incremental output must match full recomputation.

---

<details>
  <summary>💡 Hint</summary>

  - At decode time, `q` has shape `(batch, 1, d_model)` (single new token).
  - Use `torch.cat([cache_k, new_k], dim=2)` to grow the cache along the seq dimension.
  - The output at decode time is `(batch, 1, d_model)` — attention over all cached tokens for the new query.
  - For validation, run the full sequence through non-cached attention and compare the last token's output.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class CachedMultiHeadAttention(nn.Module):
    """
    Multi-Head Attention with KV-Cache for efficient autoregressive generation.
    """
    def __init__(self, num_heads: int, d_model: int):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cache_k=None, cache_v=None):
        """
        Args:
            x (Tensor): Input of shape (batch, seq_len, d_model)
                        seq_len=full for prefill, seq_len=1 for decode
            cache_k (Tensor, optional): Cached keys (batch, num_heads, cached_len, d_head)
            cache_v (Tensor, optional): Cached values (batch, num_heads, cached_len, d_head)

        Returns:
            output (Tensor): (batch, seq_len, d_model)
            new_cache_k (Tensor): Updated key cache
            new_cache_v (Tensor): Updated value cache
        """
        batch_size, seq_len, _ = x.shape

        # Project Q, K, V for the new tokens only
        Q = self.Q_w(x)  # (batch, seq_len, d_model)
        K = self.K_w(x)
        V = self.V_w(x)

        # Reshape to (batch, num_heads, seq_len, d_head)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        # Append to cache
        if cache_k is not None:
            K = torch.cat([cache_k, K], dim=2)  # (batch, heads, cached_len + seq_len, d_head)
            V = torch.cat([cache_v, V], dim=2)

        # Save updated cache
        new_cache_k = K
        new_cache_v = V

        # Attention: Q attends to full K/V (cached + new)
        # Q: (batch, heads, seq_len, d_head)
        # K: (batch, heads, total_len, d_head)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)  # (batch, heads, seq_len, d_head)

        # Combine heads
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

        return self.W_out(output), new_cache_k, new_cache_v

## 📚 Understanding KV-Cache

### Without Cache (Naive Autoregressive)

```
Step 1: Compute attention over [t1]              → 1 token
Step 2: Compute attention over [t1, t2]          → 2 tokens (recompute t1!)
Step 3: Compute attention over [t1, t2, t3]      → 3 tokens (recompute t1, t2!)
...
Step T: Compute attention over [t1, ..., tT]     → T tokens

Total projections: 1 + 2 + 3 + ... + T = O(T²)
```

### With KV-Cache

```
Step 1 (prefill):  Project Q,K,V for [t1]        → cache K1, V1
Step 2 (decode):   Project Q,K,V for [t2] only   → cache [K1,K2], [V1,V2]
Step 3 (decode):   Project Q,K,V for [t3] only   → cache [K1,K2,K3], [V1,V2,V3]
...
Step T (decode):   Project Q,K,V for [tT] only   → attend Q_T over all cached K,V

Total projections: 1 + 1 + 1 + ... + 1 = O(T)
```

### Memory Cost

The tradeoff: KV-Cache uses **memory** to save **compute**.

```
Cache size per layer = 2 (K+V) × batch × num_kv_heads × seq_len × d_head × dtype_bytes

Example: Llama 2 7B (32 layers, 32 heads, d_head=128, fp16)
  Per layer:  2 × batch × 32 × seq_len × 128 × 2 bytes
  32 layers:  2 × batch × 32 × seq_len × 128 × 2 × 32 = 512 MB for seq_len=2048, batch=1
```

This is why MQA/GQA are so important — they reduce `num_kv_heads` and thus cache size.

### Prefill vs Decode

| Phase | Input | Operation | Compute |
|-------|-------|-----------|--------|
| Prefill | Full prompt (N tokens) | Process all tokens at once | GPU-bound (parallel) |
| Decode | 1 token at a time | Attend to cached K/V | Memory-bound (sequential) |

In [ ]:
# Validate: incremental decode must match full recomputation
torch.manual_seed(42)
batch_size, seq_len, d_model, num_heads = 2, 6, 8, 2
tokens = torch.randn(batch_size, seq_len, d_model)

attn = CachedMultiHeadAttention(num_heads, d_model)
attn.eval()

# Method 1: Full forward (no cache) — the reference
with torch.no_grad():
    full_output, _, _ = attn(tokens)

# Method 2: Incremental with KV-Cache — token by token
cache_k, cache_v = None, None
incremental_outputs = []
with torch.no_grad():
    for t in range(seq_len):
        token_t = tokens[:, t:t+1, :]  # (batch, 1, d_model)
        out_t, cache_k, cache_v = attn(token_t, cache_k, cache_v)
        incremental_outputs.append(out_t)

# The last incremental output should match the last position of full output
last_full = full_output[:, -1:, :]
last_incremental = incremental_outputs[-1]

print(f"Full output (last token):        {last_full[0, 0, :4]}")
print(f"Incremental output (last token): {last_incremental[0, 0, :4]}")

assert torch.allclose(last_full, last_incremental, atol=1e-6)
print(f"\nCache shape: K={cache_k.shape}, V={cache_v.shape}")
print("\n✅ KV-Cache implementation is correct!")

In [ ]:
# Bonus: Verify ALL positions match (not just last)
# Each incremental output at step t should match full_output[:, t, :]
for t in range(seq_len):
    expected = full_output[:, t:t+1, :]
    actual = incremental_outputs[t]
    match = torch.allclose(expected, actual, atol=1e-6)
    print(f"  Step {t}: {'✅' if match else '❌'}")

print("\n✅ All positions match between cached and full computation!")


---

# 14 · Transformer Decoder Block

*Source: `torch/transformer/14-Transformer-Decoder-Block/transformer-decoder-block.ipynb`*

---


# Implement a Transformer Decoder Block from Scratch
### Problem Statement
The Transformer Decoder Block is the core repeating unit in decoder-only LLMs (GPT, Llama, Mistral, etc.). Each block combines all the components you've built in this series:

```
x → RMSNorm → Multi-Head Attention → + (residual) → RMSNorm → SwiGLU FFN → + (residual) → output
```

This is the **Pre-Norm** architecture used by modern LLMs. A full model stacks N of these blocks (e.g., 32 for Llama 2 7B).

Your goal is to implement a complete decoder block using the components from previous exercises, with causal masking for autoregressive generation.

---

### Requirements

1. **Components**
   - RMSNorm (from `08-RMS-Norm`)
   - Multi-Head Attention (from `04-Multi-Head-Attention`) with **causal mask**
   - SwiGLU Feed-Forward Network (from `10-Feed-Forward-Network`)
   - Residual connections (from `09-Residual-Connection`)

2. **Causal Masking**
   - Generate a lower-triangular mask so each token can only attend to itself and previous tokens.
   - This enforces the autoregressive property.

3. **Architecture (Pre-Norm)**
   ```
   h = x + Attention(RMSNorm(x), causal_mask)
   output = h + FFN(RMSNorm(h))
   ```

4. **Validate**
   - Output shape must be `(batch_size, seq_len, d_model)`.
   - Causal mask must prevent future token leakage.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Pre-Norm architecture (norm before sublayer, not after).
- ✅ Causal masking must be correct.
- ✅ All sub-components should be implemented from scratch (no `nn.TransformerDecoderLayer`).

---

<details>
  <summary>💡 Hint</summary>

  - Causal mask: `torch.tril(torch.ones(seq_len, seq_len))` — lower triangular matrix of ones.
  - In the attention scores, mask positions with 0 should be filled with `-inf` before softmax.
  - The block has no learnable parameters of its own — it just orchestrates sub-modules.
  - Remember: Pre-Norm means `x + sublayer(norm(x))`, not `norm(x + sublayer(x))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
# Building blocks (from previous exercises)

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x / rms) * self.scale

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention with causal mask support."""
    def __init__(self, num_heads, d_model):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.d_head = d_model // num_heads

        self.Q_w = nn.Linear(d_model, d_model, bias=False)
        self.K_w = nn.Linear(d_model, d_model, bias=False)
        self.V_w = nn.Linear(d_model, d_model, bias=False)
        self.W_out = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        """Self-attention: q=k=v=x"""
        batch_size, seq_len, _ = x.shape

        Q = self.Q_w(x)
        K = self.K_w(x)
        V = self.V_w(x)

        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_head ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        output = torch.matmul(attn_weights, V)

        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_out(output)

In [ ]:
class SwiGLUFeedForward(nn.Module):
    """SwiGLU FFN."""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or int(2 / 3 * 4 * d_model)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))

In [ ]:
def create_causal_mask(seq_len):
    """Returns a lower-triangular boolean mask of shape (1, 1, seq_len, seq_len)."""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len) for broadcasting

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    A single Transformer Decoder Block (Pre-Norm architecture).
    x → RMSNorm → Attention → + residual → RMSNorm → FFN → + residual → output
    """
    def __init__(self, num_heads: int, d_model: int, d_ff: int = None):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads, d_model)
        self.ffn = SwiGLUFeedForward(d_model, d_ff)
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)

    def forward(self, x, mask=None):
        """
        Args:
            x (Tensor): Input of shape (batch_size, seq_len, d_model)
            mask (Tensor, optional): Causal mask

        Returns:
            Tensor: Output of shape (batch_size, seq_len, d_model)
        """
        # Pre-Norm: norm before sublayer, residual on clean path
        h = x + self.attn(self.norm1(x), mask)   # attention + residual
        output = h + self.ffn(self.norm2(h))      # FFN + residual
        return output

## 📚 The Complete Decoder Block

### Architecture Diagram

```
        ┌───────────────────────────────────┐
        │                                   │
  x ────┤─→ RMSNorm ─→ Multi-Head Attn ─→ + ─→ h
        │        (causal mask)              │
        └───────────────────────────────────┘
        ┌───────────────────────────────────┐
        │                                   │
  h ────┤─→ RMSNorm ─→ SwiGLU FFN ──────→ + ─→ output
        │                                   │
        └───────────────────────────────────┘
```

### How It Maps to Real Models

| Component | Original Transformer | Llama 2/3 | GPT-2 |
|-----------|---------------------|-----------|-------|
| Norm | LayerNorm (Post) | RMSNorm (Pre) | LayerNorm (Pre) |
| Attention | MHA | GQA | MHA |
| FFN | ReLU | SwiGLU | GELU |
| Position | Sinusoidal | RoPE | Learned |
| Bias | Yes | No | Yes |

### Causal Mask Visualization

For seq_len=4, the causal mask looks like:
```
  t1 t2 t3 t4
t1 [1  0  0  0]   ← t1 can only see t1
t2 [1  1  0  0]   ← t2 can see t1, t2
t3 [1  1  1  0]   ← t3 can see t1, t2, t3
t4 [1  1  1  1]   ← t4 can see all tokens
```

Positions with 0 are filled with `-inf` before softmax, making their attention weight exactly 0.

### Parameter Breakdown

For a block with `d_model=4096, num_heads=32, d_ff=11008` (Llama 2 7B):

| Component | Parameters |
|-----------|----------|
| Attention (Q,K,V,O) | 4 × 4096² = 67M |
| FFN (gate, up, down) | 3 × 4096 × 11008 = 135M |
| RMSNorm × 2 | 2 × 4096 = 8K |
| **Block total** | **~202M** |
| **32 blocks** | **~6.5B** (+ embeddings = ~7B) |

In [ ]:
# Test
torch.manual_seed(42)
batch_size, seq_len, d_model, num_heads = 2, 6, 32, 4

block = TransformerDecoderBlock(num_heads, d_model)
x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len)

output = block(x, mask)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Mask shape:   {mask.shape}")
assert output.shape == x.shape

# Count parameters
total_params = sum(p.numel() for p in block.parameters())
print(f"\nTotal parameters: {total_params:,}")
print("\n✅ Transformer Decoder Block works correctly!")

In [ ]:
# Verify causal masking: changing a future token should NOT affect earlier outputs
torch.manual_seed(42)
x1 = torch.randn(1, 4, d_model)
x2 = x1.clone()
x2[:, 3, :] = torch.randn(d_model)  # Change the last token

mask = create_causal_mask(4)
block.eval()
with torch.no_grad():
    out1 = block(x1, mask)
    out2 = block(x2, mask)

# First 3 tokens should be identical (they can't see token 4)
print("Outputs match for tokens that can't see the changed future token:")
for t in range(4):
    match = torch.allclose(out1[:, t, :], out2[:, t, :], atol=1e-6)
    changed = "(changed)" if t == 3 else ""
    print(f"  Token {t}: {'✅ match' if match else '❌ different'} {changed}")

print("\n✅ Causal masking verified — future tokens don't leak!")

In [ ]:
# Bonus: Stack multiple blocks to form a mini decoder
num_layers = 4
blocks = nn.ModuleList([TransformerDecoderBlock(num_heads, d_model) for _ in range(num_layers)])

h = x
for blk in blocks:
    h = blk(h, mask)

print(f"Input:  {x.shape}")
print(f"After {num_layers} blocks: {h.shape}")
total = sum(p.numel() for p in blocks.parameters())
print(f"Total params ({num_layers} layers): {total:,}")